In [1]:

import random

# Public Transport Scheduling using Genetic Algorithm

TOTAL_TIME = 120       # Operating duration in minutes
NUM_BUSES = 6           # Number of bus departures
POPULATION_SIZE = 30
GENERATIONS = 100

# Create a random schedule of positive time intervals
def create_schedule():
    gaps = [1] * NUM_BUSES

    for _ in range(TOTAL_TIME - NUM_BUSES):
        gaps[random.randrange(NUM_BUSES)] += 1

    return gaps

# Fitness: lower passenger waiting time is better
def fitness(schedule):
    return sum(gap * gap for gap in schedule)

# Select the better schedule from two random candidates
def selection(population):
    a, b = random.sample(population, 2)
    return a if fitness(a) < fitness(b) else b

# Crossover: combine two schedules and preserve total time
def crossover(p1, p2):
    child = [(a + b) // 2 for a, b in zip(p1, p2)]

    # Adjust intervals so their sum remains TOTAL_TIME
    difference = TOTAL_TIME - sum(child)

    while difference > 0:
        i = random.randrange(NUM_BUSES)
        child[i] += 1
        difference -= 1

    while difference < 0:
        choices = [i for i in range(NUM_BUSES) if child[i] > 1]
        i = random.choice(choices)
        child[i] -= 1
        difference += 1

    return child

# Mutation: move one minute between two intervals
def mutation(schedule):
    child = schedule[:]

    if random.random() < 0.3:
        i, j = random.sample(range(NUM_BUSES), 2)

        if child[i] > 1:
            child[i] -= 1
            child[j] += 1

    return child

# Initialize population
population = [create_schedule()
              for _ in range(POPULATION_SIZE)]

# Run the Genetic Algorithm
for generation in range(GENERATIONS):
    new_population = []

    # Preserve the best solution (elitism)
    best = min(population, key=fitness)
    new_population.append(best[:])

    while len(new_population) < POPULATION_SIZE:
        parent1 = selection(population)
        parent2 = selection(population)

        child = crossover(parent1, parent2)
        child = mutation(child)

        new_population.append(child)

    population = new_population

# Find the best schedule
best_schedule = min(population, key=fitness)

# Calculate expected waiting time
average_wait = fitness(best_schedule) / (2 * TOTAL_TIME)

# Convert intervals into departure times
departure_times = []
time = 0

for gap in best_schedule:
    departure_times.append(time)
    time += gap

print("Optimized Bus Schedule")
print("Operating duration:", TOTAL_TIME, "minutes")
print("Number of buses:", NUM_BUSES)
print("Departure intervals:", best_schedule)
print("Departure times:", departure_times)
print("Expected average waiting time:",
      round(average_wait, 2), "minutes")


Optimized Bus Schedule
Operating duration: 120 minutes
Number of buses: 6
Departure intervals: [20, 20, 20, 20, 20, 20]
Departure times: [0, 20, 40, 60, 80, 100]
Expected average waiting time: 10.0 minutes
